# LeetCode 69 — Sqrt(x)
Nhập môn Python, giảng từng bước

**Độ khó:** Easy · **Kỹ thuật trọng tâm:** Binary Search · số nguyên · bất biến  
**Bài gốc:** https://leetcode.com/problems/sqrtx/

## Bạn sẽ học được gì?
1. Hiểu căn bậc hai nguyên và cách làm tròn xuống.
2. Tự cài cách thử tuần tự rồi nhận ra điểm nghẽn.
3. Giải thích vì sao Binary Search rút ngắn phạm vi tìm.
4. Trace biến left, right, mid và nộp đúng mySqrt.
5. Tự làm bài tập biến thể và phân tích Time/Space Complexity.

**Cách học:** đọc Markdown → đoán kết quả của cell code tiếp theo → chạy cell → so sánh. Phần TODO chỉ để luyện tập, không tự chạy bài tập chưa hoàn thiện.

## 1. Đề bài — ta cần tìm số nào?

Cho **số nguyên không âm** `x`. Trả về **phần nguyên của căn bậc hai** của `x`, tức số nguyên lớn nhất `k` sao cho `k*k <= x`.

| x | Vì sao? | Kết quả |
|---:|---|---:|
| 4 | 2×2=4 | 2 |
| 8 | 2×2=4 ≤ 8, nhưng 3×3=9 > 8 | 2 |
| 0 | 0×0=0 | 0 |
| 15 | 3×3=9, 4×4=16>15 | 3 |

**Lưu ý:** đề không yêu cầu số thập phân như 2.828...; không dùng `sqrt`/lũy thừa phân số trong lời giải nộp.

In [ ]:
print("x=4: 2*2 <= 4?", 2*2 <= 4, "; 3*3 > 4?", 3*3 > 4)
print("x=8: 2*2 <= 8?", 2*2 <= 8, "; 3*3 > 8?", 3*3 > 8)
print("x=15: 3*3 <= 15?", 3*3 <= 15, "; 4*4 > 15?", 4*4 > 15)
print("x=0: căn nguyên = 0 vì 0*0 =", 0*0)

## 2. Python nền tảng: `*`, `**`, `//` và `<=`

- `k * k`: nhân `k` với chính nó.
- `k ** 2`: lũy thừa bậc hai, cùng ý nghĩa với `k*k`.
- `a // b`: chia **lấy phần nguyên xuống** (với các số không âm trong bài).
- `k*k <= x`: một **điều kiện logic** trả `True` hoặc `False`.

Một `mid` phải là **số nguyên** để chỉ đúng ứng viên cần thử.

In [ ]:
k, x = 3, 10
print("k*k =", k * k, "k**2 =", k ** 2)
print("3*3 <= 10 ?", k*k <= x)
print("(2 + 9) // 2 =", (2 + 9) // 2)
print("(2 + 9) / 2  =", (2 + 9) / 2, "(kiểu float)")

## 3. Cách dễ nghĩ nhất: thử từng k từ 0 trở đi

Ta thử `0,1,2,...` cho đến khi gặp số đầu tiên có bình phương **lớn hơn x**. Khi đó kết quả là số ngay trước nó.

**Vì sao đúng?** Bình phương các số nguyên không âm tăng dần. Nếu `k*k > x` thì mọi số lớn hơn `k` cũng không hợp lệ.

Ta dùng `while`: mỗi vòng kiểm tra điều kiện; `k += 1` nghĩa là tăng `k` lên 1.

In [ ]:
def sqrt_linear(x: int) -> int:
    k = 0
    while (k + 1) * (k + 1) <= x:
        k += 1
    return k

for x in [0, 1, 2, 8, 9, 15, 16]:
    print(x, "->", sqrt_linear(x))

## 4. Trace cách tuần tự bằng tay

Với `x=15`, lần lượt kiểm tra **ứng viên mới** `k+1`:

| k hiện tại | Ứng viên k+1 | Bình phương | Còn đi tiếp? |
|---:|---:|---:|---|
| 0 | 1 | 1 | Có |
| 1 | 2 | 4 | Có |
| 2 | 3 | 9 | Có |
| 3 | 4 | 16 | Không → trả 3 |

Phương pháp này làm khoảng `√x` phép thử. Nếu `x` rất lớn, có thể phải thử hàng chục nghìn lần.

In [ ]:
def trace_sqrt_linear(x):
    k = 0
    while True:
        candidate = k + 1
        allowed = candidate * candidate <= x
        print(f"k={k} | thử {candidate}²={candidate*candidate} | <= {x}? {allowed}")
        if not allowed:
            print("DỪNG ->", k)
            return k
        k += 1

trace_sqrt_linear(15)

## 5. Bước đột phá: các ứng viên chia thành hai phía

Hàm điều kiện `k*k <= x` có tính chất **đơn điệu (monotonic)**: khi `k` tăng, trạng thái chỉ có thể đi từ `True` sang `False`, không quay ngược.

Ví dụ `x=15`: `0:T, 1:T, 2:T, 3:T, 4:F, 5:F, ...`.

Ta muốn tìm **số True cuối cùng**. Thay vì duyệt mọi giá trị, hãy kiểm tra số giữa rồi loại bỏ **một nửa** phạm vi.

In [ ]:
x = 15
for k in range(0, 8):
    print(k, "->", k*k <= x)

## 6. Binary Search là gì?

**Tìm kiếm nhị phân (Binary Search)** làm việc trên một phạm vi đã có quy luật đơn điệu. Giữ hai biên **bao gồm** `left`, `right`, rồi xét `mid = (left + right) // 2`.

- Nếu `mid*mid <= x`: `mid` hợp lệ → ghi `answer=mid`, thử tìm số **lớn hơn** bằng `left=mid+1`.
- Nếu `mid*mid > x`: `mid` quá lớn → tìm **nhỏ hơn** bằng `right=mid-1`.

Hai biên vượt nhau (`left > right`) thì dừng. Đây là quy tắc **không bỏ lỡ** ứng viên.

In [ ]:
left, right = 0, 15
mid = (left + right) // 2
print("Phạm vi:", left, right)
print("Điểm giữa:", mid)
print("mid²:", mid*mid, "<=" ,15, "?",mid*mid<=15)
# Không hợp lệ: loại mid và toàn bộ đoạn [mid, right].
right = mid - 1
print("Phạm vi còn lại:", left, right)

## 7. Code đầy đủ — nhị phân tìm phần nguyên căn

Ta khởi đầu `answer=0`, chắc chắn hợp lệ vì `x` không âm. Cách làm `left=0, right=x` áp dụng được cả `x=0` và `x=1`.

Hãy đọc **điều kiện trước, cập nhật sau**, không ghi nhớ máy móc.

In [ ]:
def sqrt_binary(x: int) -> int:
    left, right = 0, x
    answer = 0
    while left <= right:
        mid = (left + right) // 2
        if mid * mid <= x:
            answer = mid               # mid là một ứng viên hợp lệ
            left = mid + 1             # thử ứng viên lớn hơn
        else:
            right = mid - 1            # bỏ phần quá lớn
    return answer

for x in [0, 1, 4, 8, 15, 16, 25]:
    print(f"sqrt_binary({x}) = {sqrt_binary(x)}")

## 8. Bảng trace cho `x=15`

| Lượt | left trước | right trước | mid | mid² | Hành động | answer sau |
|---:|---:|---:|---:|---:|---|---:|
| 1 | 0 | 15 | 7 | 49 | Quá lớn → right=6 | 0 |
| 2 | 0 | 6 | 3 | 9 | Hợp lệ → answer=3; left=4 | 3 |
| 3 | 4 | 6 | 5 | 25 | Quá lớn → right=4 | 3 |
| 4 | 4 | 4 | 4 | 16 | Quá lớn → right=3 | 3 |

Lúc này `left=4 > right=3`, trả `3`. Bảng ghi **biên trước lần cập nhật** để tránh lẫn trạng thái.

In [ ]:
def trace_sqrt_binary(x):
    left, right, answer, step = 0, x, 0, 0
    print("x =", x)
    while left <= right:
        step += 1
        mid = (left + right) // 2
        ok = mid * mid <= x
        print(f"{step}: L={left}, R={right}, mid={mid}, mid²={mid*mid}, hợp lệ={ok}, ans trước={answer}")
        if ok:
            answer = mid
            left = mid + 1
        else:
            right = mid - 1
        print(f"   sau cập nhật: L={left}, R={right}, ans={answer}")
    print("TRẢ VỀ", answer)
    return answer

trace_sqrt_binary(15)
trace_sqrt_binary(8)

## 9. Bất biến (invariant): vì sao không trả nhầm?

**Bất biến:** `answer` luôn là ứng viên đã kiểm tra và thỏa `answer² <= x`. Những số bị loại ở **phía trên** đều chắc chắn quá lớn; những ứng viên hợp lệ đã thử được giữ trong `answer`.

Khi `mid` hợp lệ, mọi số ≤ `mid` không thể tốt hơn `mid`, nên ta tìm ở bên phải. Khi `mid` không hợp lệ, mọi số ≥ `mid` đều sai, nên ta thu hẹp sang trái. Khi hết phạm vi, không còn ứng viên nào lớn hơn đáp án mà chưa bị chứng minh sai.

**Vì sao vòng lặp dừng?** Mỗi vòng đẩy `left` lên ít nhất 1 hoặc `right` xuống ít nhất 1.

## 10. Nộp LeetCode — chỉ sao chép lớp `Solution`

LeetCode yêu cầu phương thức **`mySqrt(self, x: int) -> int`**, không phải `sqrt_binary`. Lớp bên dưới **không** dùng `math.sqrt`, dấu căn, hoặc lũy thừa phân số.

In [ ]:
class Solution:
    def mySqrt(self, x: int) -> int:
        left, right = 0, x
        answer = 0
        while left <= right:
            mid = (left + right) // 2
            if mid * mid <= x:
                answer = mid
                left = mid + 1
            else:
                right = mid - 1
        return answer

## 11. Test ví dụ, biên và giá trị lớn

`assert` phát hiện kết quả sai ngay lập tức. `math.isqrt(x)` là hàm **đối chiếu độc lập** khi kiểm thử, **không xuất hiện trong lời giải nộp**.

In [ ]:
import math
cases_69 = [(0,0),(1,1),(2,1),(3,1),(4,2),(8,2),(9,3),(15,3),
            (16,4),(24,4),(25,5),(26,5),(2147395599,46339),(2147483647,46340)]
for x, expected in cases_69:
    actual = Solution().mySqrt(x)
    assert actual == expected == math.isqrt(x), (x, actual, expected)
print("PASS",len(cases_69),"ca mẫu / biên")

## 12. Đối chiếu toàn bộ số nhỏ với cách tuần tự

Chạy hai cách độc lập trên `0..999` giúp phát hiện lỗi lệch biên, ví dụ quên cập nhật `left=mid+1`.

In [ ]:
for x in range(1000):
    assert sqrt_binary(x) == sqrt_linear(x) == math.isqrt(x)
print("PASS 1000/1000 ca đối chiếu")

## 13. Độ phức tạp

Gọi `x` là **giá trị số nguyên đầu vào**.

| Cách | Time | Extra Space |
|---|---|---|
| Thử tuần tự | O(√x + 1) | O(1) |
| Binary Search | O(log(x+1)) | O(1) |

Trong mỗi vòng Binary Search, miền ứng viên giảm xấp xỉ một nửa. Đây là lý do log xuất hiện, không phải vì code có ít dòng. Đánh giá số vòng cho các ví dụ lớn để thấy rõ.

In [ ]:
def count_binary_steps(x):
    l, r, steps = 0, x, 0
    while l <= r:
        mid = (l + r) // 2
        steps += 1
        if mid*mid <= x: l = mid + 1
        else: r = mid - 1
    return steps

for x in [10, 1000, 1000000, 2147483647]:
    print("x=",x,"-> số lượt Binary Search=",count_binary_steps(x))

## 14. Sai lầm thường gặp

1. Dùng `/` thay `//` trong phép tính mid → `mid` là `float`, không còn chỉ số nguyên.
2. Khi `mid² <= x`, trả ngay `mid` → có thể bỏ qua kết quả lớn hơn.
3. Khi mid đúng/sai, chỉ đặt `left=mid` hoặc `right=mid` → có thể **lặp mãi** tại biên.
4. Đoán `round(x**0.5)` → làm tròn sai yêu cầu; số thực có sai số với số lớn.
5. Trong một số ngôn ngữ có số nguyên cố định, `mid*mid` có nguy cơ tràn; Python integer không bị tràn kiểu 32-bit.

In [ ]:
x = 8
print("Làm tròn gần nhất:",round(x**0.5),"(sai nếu trả 3)")
print("Cần làm tròn xuống:",sqrt_binary(x))
print("Mid phép /:",(0+8)/2,"; Mid phép //:",(0+8)//2)

## 15. Mở rộng: tránh phép nhân `mid*mid`

Với `mid > 0`, điều kiện `mid² <= x` tương đương `mid <= x // mid` (trong miền nguyên không âm). Phiên bản dưới dùng `left=1` và xử lý `x=0` riêng để **không chia cho 0**.

Ý tưởng này hữu ích ở ngôn ngữ mà bình phương có thể vượt phạm vi kiểu nguyên.

In [ ]:
def sqrt_without_square(x: int) -> int:
    if x == 0:
        return 0
    left, right, answer = 1, x, 0
    while left <= right:
        mid = (left + right) // 2
        if mid <= x // mid:
            answer = mid
            left = mid + 1
        else:
            right = mid - 1
    return answer

assert all(sqrt_without_square(x)==sqrt_binary(x) for x in range(1000))
print("PASS phiên bản dùng phép chia")

## 16. Nhận diện pattern

Nếu đề yêu cầu **giá trị nguyên lớn nhất/nhỏ nhất thỏa điều kiện đơn điệu**, hãy nghĩ đến **Binary Search trên đáp án**. Điều cần chứng minh trước tiên là: “Nếu một ứng viên hợp lệ/không hợp lệ, ta loại được toàn bộ phía nào?”

## 17. Năm bài tập tự luyện

**Hướng dẫn:** tự tính tay trước; mở gợi ý theo từng cấp. Chỉ xem đáp án tham khảo sau cả năm bài.

### Bài 1 — Đoán kết quả toán tử · Tự làm

Với `k=5`, trả tuple `(k*k, k**2, 11//2)`.

**Gợi ý cấp 1 — tư duy:** Nhân cùng một số hai lần.

<details><summary>Gợi ý cấp 2 — cú pháp</summary>

Dùng toán tử `*`, `**`, `//`.

</details>

<details><summary>Gợi ý cấp 3 — gần-code</summary>

Viết `return (k*k, k**2, 11//2)`.

</details>

In [ ]:
def exercise_69_1(k: int):
    # TODO: return tuple gồm 3 kết quả
    pass

# Tự mở comment sau khi hoàn thiện:
# assert exercise_69_1(5) == (25,25,5)

### Bài 2 — Tính tay · Tự làm

Tìm căn nguyên làm tròn xuống của 26, chỉ trả số nguyên.

**Gợi ý cấp 1 — tư duy:** Xét các bình phương quanh 26.

<details><summary>Gợi ý cấp 2 — cú pháp</summary>

5²=25; 6²=36.

</details>

<details><summary>Gợi ý cấp 3 — gần-code</summary>

Kết quả là 5.

</details>

In [ ]:
def exercise_69_2() -> int:
    # TODO: điền đáp án tính tay
    pass

# Tự mở comment sau khi hoàn thiện:
# assert exercise_69_2() == 5

### Bài 3 — Viết cách tuần tự · Tự làm

Trả `floor(sqrt(x))` bằng cách thử từ k=0, không dùng thư viện.

**Gợi ý cấp 1 — tư duy:** Miễn `(k+1)² <= x` thì tiếp tục.

<details><summary>Gợi ý cấp 2 — cú pháp</summary>

Dùng `while` và `k += 1`.

</details>

<details><summary>Gợi ý cấp 3 — gần-code</summary>

Mẫu: khởi tạo k=0; while (k+1)*(k+1)<=x: k+=1.

</details>

In [ ]:
def exercise_69_3(x: int) -> int:
    # TODO: thử các ứng viên nguyên
    pass

# Tự mở comment sau khi hoàn thiện:
# assert exercise_69_3(17) == 4

### Bài 4 — Tự viết Binary Search · Tự làm

Không gọi `sqrt_binary`; tự tìm số nguyên lớn nhất có bình phương ≤ x.

**Gợi ý cấp 1 — tư duy:** Nếu mid hợp lệ, tìm sang phải.

<details><summary>Gợi ý cấp 2 — cú pháp</summary>

left=0, right=x, mid=(left+right)//2.

</details>

<details><summary>Gợi ý cấp 3 — gần-code</summary>

Nhớ `answer=mid; left=mid+1` hoặc `right=mid-1`.

</details>

In [ ]:
def exercise_69_4(x: int) -> int:
    # TODO: binary search
    pass

# Tự mở comment sau khi hoàn thiện:
# assert exercise_69_4(99) == 9

### Bài 5 — Biến thể: kiểm tra số chính phương · Tự làm

Trả `True` nếu x có căn bậc hai nguyên, `False` nếu không.

**Gợi ý cấp 1 — tư duy:** Một số chính phương bằng `k*k`.

<details><summary>Gợi ý cấp 2 — cú pháp</summary>

Bạn có thể tự tính k bằng bài 4.

</details>

<details><summary>Gợi ý cấp 3 — gần-code</summary>

`k = sqrt_binary(x); return k*k == x` (bản tham khảo).

</details>

In [ ]:
def exercise_69_5(x: int) -> bool:
    # TODO: True nếu x là số chính phương
    pass

# Tự mở comment sau khi hoàn thiện:
# assert exercise_69_5(49) and not exercise_69_5(50)

## 18. Đáp án tham khảo — chỉ đọc sau khi tự làm

Các hàm `answer_*` dùng tên riêng, không ghi đè các hàm bài tập. Chúng được kiểm tra độc lập dưới đây.

In [ ]:
def answer_69_1(k): return k*k, k**2, 11//2

def answer_69_2(): return 5

def answer_69_3(x):
    k=0
    while (k+1)*(k+1)<=x:
        k+=1
    return k

def answer_69_4(x):
    left,right,answer=0,x,0
    while left<=right:
        mid=(left+right)//2
        if mid*mid<=x:
            answer=mid
            left=mid+1
        else:
            right=mid-1
    return answer

def answer_69_5(x):
    k=answer_69_4(x)
    return k*k==x

assert answer_69_1(5)==(25,25,5)
assert answer_69_2()==5
assert answer_69_3(17)==4
assert answer_69_4(99)==9
assert answer_69_5(49) and not answer_69_5(50)
print("PASS tất cả đáp án bài tập #69")

## Tóm tắt — mẫu tư duy cần nhớ

**Số nguyên lớn nhất `k` có `k² ≤ x`** → điều kiện đơn điệu → Binary Search → hợp lệ thì ghi đáp án và đi sang phải; không hợp lệ thì đi sang trái. **Time O(log(x+1)), extra space O(1).**

## Checklist trước khi submit

- [ ] Tôi phân biệt căn bậc hai thực và căn nguyên **làm tròn xuống**.
- [ ] Tôi dùng `//` để tính `mid`.
- [ ] Tôi giải thích được vì sao bỏ được nửa miền khi kiểm tra một `mid`.
- [ ] Tôi đã thử `x=0,1,8,16,2147483647`.
- [ ] Tôi chỉ dán `class Solution` với tên **`mySqrt`**.